# 05 · Classification: LGG vs GBM

**Task.** Predict the grade (`Grade`: LGG = 0, GBM = 1) from age, gender, race and the 20 mutations. `Primary_Diagnosis` and `Project` are **excluded** (they encode the grade: notebooks 01 and 02). GBM is the **positive class**: sensitivity = share of GBM patients detected, specificity = share of LGG patients correctly identified.

**Protocol** (same as the regression, see `docs/DECISIONS.md`):
- **Nested cross-validation**: 10 **stratified** outer folds (each fold keeps the 58% / 42% class proportions) and 3 stratified inner folds with `GridSearchCV`; seed 42; preprocessing inside each `Pipeline`.
- **Inner selection criterion: ROC-AUC**, which does not depend on a decision threshold.
- **Reported metrics** on the out-of-fold predicted probabilities: ROC-AUC, and at the default threshold 0.5: accuracy, sensitivity, specificity and balanced accuracy. Baseline: a model that always predicts the class proportions of its training part (AUC = 0.5 within each fold; its accuracy equals the majority share). Its *pooled* AUC is not exactly 0.5 because the constant differs slightly between folds, so ranking the pooled predictions is not completely random.
- **95% bootstrap intervals**, stratified by class (2000 resamples, the same for every model). **Rule:** no model is declared better than another if their intervals overlap; a paired bootstrap of the AUC difference is reported as a complementary check.
- **Final models** for interpretation: the same tuning procedure applied once to all patients.

| # | Section |
|---|---|
| 1 | Setup |
| 2 | Models and hyperparameter grids |
| 3 | Nested cross-validation |
| 4 | Model comparison and ROC curves |
| 5 | Chosen hyperparameters across folds |
| 6 | Confusion matrices and decision threshold |
| 7 | Logistic regression: coefficients and odds ratios |
| 8 | Naive Bayes: what the model learns |
| 9 | Pruned decision tree |
| 10 | Permutation importance |
| 11 | Summary |

In [ ]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Project root (folder containing src/config.py) not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, confusion_matrix
from IPython.display import display

from src import config, data, evaluation as ev, models, plotting, reporting
from src.pipelines import format_value, readable_names
from src.plotting import CLASS_COLORS, NEUTRAL

plotting.set_style()
pd.set_option("display.precision", 3)

df = data.load_processed()
FEATURES = data.feature_columns("classification")
assert not set(FEATURES) & set(config.LEAKAGE_COLS + ["Grade"]), "Leakage column among the predictors"
X = df[FEATURES]
y = df["Grade"].values
results = {"features": FEATURES, "n": len(y), "n_gbm": int(y.sum())}
print(f"{len(y)} patients ({int(y.sum())} GBM), {len(FEATURES)} predictors: {FEATURES}")

## 2. Models and hyperparameter grids

- **Logistic regression** with L1 or L2 penalty (the penalty type is itself a hyperparameter).
- **Naive Bayes**: assumes the predictors are independent given the class. All predictors are binary except age, which is cut into quantile bins (number of bins tuned) so that a Bernoulli model can be used for every column.
- **Decision tree** grown fully and then **pruned** by cost-complexity pruning (`ccp_alpha`), as recommended in the course (a tree must be pruned to avoid overfitting). In the regression notebook the tree was limited in advance (pre-pruning) instead; both approaches control the size of the tree.
- **Random forest** and **XGBoost**: ensembles of trees (bagging with random feature subsets, and boosting).
- **SVM** with linear or RBF kernel; probabilities are obtained with Platt scaling (`probability=True`) to compute the AUC.

In [ ]:
MODELS = models.classification_models(FEATURES)

def describe_grid(grid):
    grids = grid if isinstance(grid, list) else [grid]
    parts = []
    for g in grids:
        parts.append(", ".join(f"{k.split('__', 1)[1]} in [{', '.join(format_value(x) for x in v)}]"
                               for k, v in g.items()))
    return " | ".join(parts) or "none"

def n_configs(grid):
    grids = grid if isinstance(grid, list) else [grid]
    return int(sum(np.prod([len(v) for v in g.values()]) for g in grids)) if grid else 1

grids = pd.DataFrame([{"Model": n, "Hyperparameters searched": describe_grid(g), "Configurations": n_configs(g)}
                      for n, (p, g) in MODELS.items()])
with pd.option_context("display.max_colwidth", None):
    display(grids)
for name, note in models.CLASSIFICATION_GRID_NOTES.items():
    print(f"{name}: {note}")
reporting.save_table_tex(grids, "clf_models_grids", column_format="lp{10.5cm}r")

## 3. Nested cross-validation

A few minutes. Permutation importance (drop in AUC when a predictor is shuffled) is computed on every outer test fold.

In [ ]:
NESTED = {}
t0 = time.time()
for name, (pipe, grid) in MODELS.items():
    t = time.time()
    NESTED[name] = ev.nested_cv(pipe, grid, X, y, task="classification", scoring=ev.CLASSIFICATION_SCORING,
                                metric_fn=ev.classification_metrics, importance=(name != "Majority baseline"))
    m = ev.classification_metrics(y, NESTED[name]["oof"])
    print(f"{name:<24} AUC = {m['AUC']:.3f}  Bal. acc = {m['Balanced accuracy']:.3f}  "
          f"Sens = {m['Sensitivity']:.3f}  Spec = {m['Specificity']:.3f}   ({time.time() - t:.0f} s)")
print(f"Total: {time.time() - t0:.0f} s")
oof = pd.DataFrame({"Case_ID": df[config.ID_COL], "Grade": y, **{n: r["oof"] for n, r in NESTED.items()}})
oof.to_csv(config.LOGS / "05_classification_oof_probabilities.csv", index=False)

## 4. Model comparison and ROC curves

In [ ]:
METRICS = ["AUC", "Balanced accuracy", "Sensitivity", "Specificity", "Accuracy"]
rows = []
for name, r in NESTED.items():
    cis = ev.metric_cis(y, r["oof"], ev.classification_metrics, stratify=y)
    row = {"Model": name}
    for metric in METRICS:
        point, lo, hi = cis[metric]
        row.update({metric: point, f"{metric} CI low": lo, f"{metric} CI high": hi,
                    f"{metric} fold mean": r["folds"][metric].mean(), f"{metric} fold SD": r["folds"][metric].std()})
    rows.append(row)
summary = pd.DataFrame(rows)
display(summary[["Model"] + [c for m in METRICS[:4] for c in (m, f"{m} CI low", f"{m} CI high")]])
results["summary"] = summary.to_dict(orient="records")

best_auc, ties_auc = ev.best_and_ties(summary, "AUC", lower_is_better=False)
best_bacc, ties_bacc = ev.best_and_ties(summary, "Balanced accuracy", lower_is_better=False)
print(f"Highest AUC: {best_auc}; models whose AUC interval overlaps it: {ties_auc}")
print(f"Highest balanced accuracy: {best_bacc}; overlapping: {ties_bacc}")
results.update({"best_auc": best_auc, "ties_auc": ties_auc, "best_bacc": best_bacc, "ties_bacc": ties_bacc})

ci = lambda m, k, d=3: f"{summary.set_index('Model').loc[m, k]:.{d}f} [{summary.set_index('Model').loc[m, k + ' CI low']:.{d}f}, {summary.set_index('Model').loc[m, k + ' CI high']:.{d}f}]"
tex = pd.DataFrame({"Model": summary["Model"], **{k: [ci(m, k) for m in summary["Model"]] for k in METRICS[:4]}})
reporting.save_table_tex(tex, "clf_model_comparison", column_format="lllll")
tex_acc = pd.DataFrame({"Model": summary["Model"], "Accuracy": [ci(m, "Accuracy") for m in summary["Model"]],
                        "AUC per fold": [f"{a:.3f} $\\pm$ {b:.3f}" for a, b in summary[["AUC fold mean", "AUC fold SD"]].values]})
reporting.save_table_tex(tex_acc, "clf_accuracy_folds", column_format="lll")

In [ ]:
auc_fn = lambda t, p: ev.auc_score(t, p)
paired = []
for name, r in NESTED.items():
    if name == "Majority baseline":
        continue
    d_best = ev.paired_difference_ci(y, r["oof"], NESTED[best_auc]["oof"], auc_fn, stratify=y) if name != best_auc else (0.0, 0.0, 0.0)
    paired.append({"Model": name, f"AUC - {best_auc}": d_best[0], "CI low": d_best[1], "CI high": d_best[2]})
paired = pd.DataFrame(paired)
display(paired)
results["paired_auc_differences"] = paired.to_dict(orient="records")
reporting.save_table_tex(pd.DataFrame({"Model": paired["Model"],
                                       f"AUC difference vs {best_auc}": [f"{a:+.3f} [{b:+.3f}, {c:+.3f}]" for a, b, c in paired.iloc[:, 1:4].values]}),
                         "clf_paired_differences", column_format="ll")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
order = summary.sort_values("AUC")["Model"].tolist()
s = summary.set_index("Model").loc[order]
for ax, metric in ((axes[0], "AUC"), (axes[1], "Balanced accuracy")):
    err = np.clip([s[metric] - s[f"{metric} CI low"], s[f"{metric} CI high"] - s[metric]], 0, None)
    ax.errorbar(s[metric], np.arange(len(order)), xerr=err, fmt="none", ecolor=NEUTRAL, elinewidth=1.4, capsize=3)
    ax.scatter(s[metric], np.arange(len(order)), color=[NEUTRAL if "baseline" in m else plotting.INK for m in order], s=45, zorder=3)
    ax.set_yticks(np.arange(len(order)), order); ax.grid(axis="y", visible=False)
    ax.set_title(f"Out-of-fold {metric} with 95% bootstrap CI")
    ax.set_xlabel(metric + (" (threshold 0.5)" if metric != "AUC" else ""))
fig.tight_layout()
plotting.save_figure(fig, "clf_model_comparison")
plt.show()

fig, ax = plt.subplots(figsize=(5.6, 5.2))
styles = ["-", "--", "-.", ":", "-", "--", "-."]
for i, (name, r) in enumerate([(n, r) for n, r in NESTED.items() if n != "Majority baseline"]):
    fpr, tpr, _ = roc_curve(y, r["oof"])
    ax.plot(fpr, tpr, lw=1.6, ls=styles[i], color=plotting.CATEGORICAL[(i + 2) % 8],
            label=f"{name} (AUC {ev.auc_score(y, r['oof']):.3f})")
ax.plot([0, 1], [0, 1], color=NEUTRAL, ls="--", lw=1, label="Chance")
ax.set_xlabel("False positive rate (1 - specificity)"); ax.set_ylabel("True positive rate (sensitivity)")
ax.set_title("Out-of-fold ROC curves"); ax.legend(fontsize=7.5, loc="lower right")
plotting.save_figure(fig, "clf_roc_curves")
plt.show()

## 5. Chosen hyperparameters across the outer folds

Same reading as in the regression: stable choices across folds mean the selection is reliable; values that change between folds mean several configurations perform about equally well.

The second table measures how **flat** the inner-CV surface of the final search is: how many configurations score within 0.005 AUC of the best one. When many do, the final choice between them is close to arbitrary (it can even be a value no outer fold chose), and the interpretation should not depend on that exact value.

In [ ]:
FINAL, SEARCH = {}, {}
hp_rows = []
for name, (pipe, grid) in MODELS.items():
    model_, params_, SEARCH[name] = ev.fit_final(pipe, grid, X, y, "classification", ev.CLASSIFICATION_SCORING,
                                                 return_search=True)
    FINAL[name] = (model_, params_)
    if not grid:
        continue
    folds = NESTED[name]["folds"]
    grids_ = grid if isinstance(grid, list) else [grid]
    for p in dict.fromkeys(k.split("__", 1)[1] for g in grids_ for k in g):
        counts = folds[p].dropna().map(format_value).value_counts() if p in folds else pd.Series(dtype=int)
        hp_rows.append({"Model": name, "Hyperparameter": p,
                        "Values chosen in the 10 outer folds": ", ".join(f"{v} ({n})" for v, n in counts.items() if v != "nan"),
                        "Final model (all data)": format_value(FINAL[name][1].get(p, "-"))})
hp = pd.DataFrame(hp_rows)
with pd.option_context("display.max_colwidth", None):
    display(hp)
reporting.save_table_tex(hp, "clf_hyperparameters", column_format="llp{7cm}l")
results["hyperparameters"] = hp.to_dict(orient="records")
results["final_hyperparameters"] = {n: p for n, (m, p) in FINAL.items()}

flat = pd.DataFrame([{"Model": n, **ev.search_flatness(s, 0.005)} for n, s in SEARCH.items() if s is not None])
display(flat)
results["final_search_flatness"] = flat.to_dict(orient="records")
reporting.save_table_tex(flat, "clf_search_flatness", float_format="{:.4f}", column_format="lrrrr")

## 6. Confusion matrices and the decision threshold

The confusion matrices use the out-of-fold probabilities and the default threshold 0.5. The threshold is a separate decision from the model: lowering it detects more GBM (higher sensitivity) at the cost of more LGG patients wrongly flagged (lower specificity). The curve below shows that trade-off for the model with the highest AUC. No threshold is tuned here: choosing it on the same out-of-fold predictions used to report performance would make the reported numbers optimistic.

In [ ]:
show = [best_auc] + [m for m in ("Logistic regression",) if m != best_auc]
fig, axes = plt.subplots(1, len(show) + 1, figsize=(4.2 * (len(show) + 1), 3.8))
for ax, name in zip(axes, show):
    cm = confusion_matrix(y, (NESTED[name]["oof"] >= ev.THRESHOLD).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap=plotting.SEQUENTIAL, cbar=False, linewidths=2, linecolor="white",
                xticklabels=["LGG", "GBM"], yticklabels=["LGG", "GBM"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{name}, threshold 0.5")
    results.setdefault("confusion_matrices", {})[name] = cm.tolist()
ax = axes[-1]
thr = np.linspace(0.05, 0.95, 91)
sens = [ev.classification_metrics(y, NESTED[best_auc]["oof"], t)["Sensitivity"] for t in thr]
spec = [ev.classification_metrics(y, NESTED[best_auc]["oof"], t)["Specificity"] for t in thr]
ax.plot(thr, sens, color=CLASS_COLORS["GBM"], label="Sensitivity (GBM detected)")
ax.plot(thr, spec, color=CLASS_COLORS["LGG"], ls="--", label="Specificity (LGG correct)")
ax.axvline(ev.THRESHOLD, color=NEUTRAL, ls=":", lw=1)
ax.set_xlabel("Decision threshold on P(GBM)"); ax.set_ylabel("Proportion"); ax.set_ylim(0, 1.02)
ax.set_title(f"{best_auc}: threshold trade-off"); ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
plotting.save_figure(fig, "clf_confusion_threshold")
plt.show()
results["threshold_curve_best"] = {"threshold": thr, "sensitivity": sens, "specificity": spec}

## 7. Logistic regression: coefficients and odds ratios

The final logistic model (penalty and C chosen on all patients; see the flatness table in Section 5) works on standardised inputs, so each coefficient is the change in the log-odds of GBM per standard deviation of the predictor, and exp(coefficient) is the corresponding **odds ratio** (> 1: higher odds of GBM). With an L1 penalty some coefficients are exactly 0.

An unpenalised logistic regression (e.g. `statsmodels`) is not used for p-values here: some mutations appear only in LGG patients (for example NOTCH1, Section 4.2 of the EDA), which causes (quasi-)complete separation and infinite maximum-likelihood coefficients. The penalty keeps the estimates finite.

In [ ]:
lr_pipe, lr_params = FINAL["Logistic regression"]
names = readable_names(lr_pipe.named_steps["prep"].get_feature_names_out())
lr_coef = pd.DataFrame({"Coefficient (log-odds per SD)": lr_pipe.named_steps["model"].coef_.ravel()}, index=names)
lr_coef["Odds ratio per SD"] = np.exp(lr_coef.iloc[:, 0])
lr_coef = lr_coef.reindex(lr_coef.iloc[:, 0].abs().sort_values(ascending=False).index)
print("Final logistic regression:", lr_params, "| coefficients equal to 0:", int((lr_coef.iloc[:, 0].abs() < 1e-10).sum()))
display(lr_coef)
results["logistic"] = {"params": lr_params, "coefficients": lr_coef.to_dict(orient="index"),
                       "n_zero": int((lr_coef.iloc[:, 0].abs() < 1e-10).sum())}
reporting.save_table_tex(lr_coef.head(12).rename_axis("Predictor").reset_index(), "clf_logistic_coefficients", float_format="{:+.3f}")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
c = lr_coef.iloc[:, 0][::-1]
ax.barh(c.index, c.values, color=[CLASS_COLORS["GBM"] if v > 0 else CLASS_COLORS["LGG"] for v in c.values], height=0.6)
ax.axvline(0, color=NEUTRAL, lw=1)
ax.set_xlabel("Coefficient (change in log-odds of GBM per SD)")
ax.set_title(f"Logistic regression ({lr_params.get('penalty', '').upper()}, C = {lr_params.get('C', 0):.3g})")
ax.grid(axis="y", visible=False)
plotting.save_figure(fig, "clf_logistic_coefficients")
plt.show()

## 8. Naive Bayes: what the model learns

Naive Bayes estimates, for every column, the probability of a 1 in each class, and combines them assuming independence given the class. The log-ratio log P(feature = 1 | GBM) / P(feature = 1 | LGG) shows the direction and strength of each column's vote. The independence assumption is clearly violated for correlated mutations (IDH1, ATRX, TP53; EDA section 8): their evidence is counted several times, which tends to push probabilities towards 0 or 1.

In [ ]:
nb_pipe, nb_params = FINAL["Naive Bayes"]
nb_names = readable_names(nb_pipe.named_steps["prep"].get_feature_names_out())
logp = nb_pipe.named_steps["model"].feature_log_prob_
nb_table = pd.DataFrame({"P(=1 | LGG)": np.exp(logp[0]), "P(=1 | GBM)": np.exp(logp[1]),
                         "log ratio GBM/LGG": logp[1] - logp[0]}, index=nb_names)
nb_table = nb_table.reindex(nb_table["log ratio GBM/LGG"].abs().sort_values(ascending=False).index)
print("Final Naive Bayes:", nb_params)
display(nb_table.head(12))
results["naive_bayes"] = {"params": nb_params, "table": nb_table.to_dict(orient="index")}
extreme = np.mean((NESTED["Naive Bayes"]["oof"] < 0.02) | (NESTED["Naive Bayes"]["oof"] > 0.98))
extreme_lr = np.mean((NESTED["Logistic regression"]["oof"] < 0.02) | (NESTED["Logistic regression"]["oof"] > 0.98))
print(f"Share of out-of-fold probabilities below 0.02 or above 0.98: Naive Bayes {extreme:.2f}, logistic {extreme_lr:.2f}")
results["naive_bayes"]["share_extreme_probabilities"] = extreme
results["logistic"]["share_extreme_probabilities"] = extreme_lr

## 9. Pruned decision tree

Cost-complexity pruning removes the branches whose contribution to purity does not compensate a penalty `ccp_alpha` per leaf. The value of `ccp_alpha` and the impurity criterion (Gini or entropy / deviance) were chosen by the inner CV on all patients. Each node shows the rule, the number of patients, the class counts [LGG, GBM] and the majority class (blue = LGG, orange = GBM, darker = purer). A split may leave the majority class unchanged in both children: it is kept because it makes the children purer (lower entropy), which improves the predicted probabilities even if not the predicted class.

In [ ]:
from sklearn.tree import plot_tree, export_text
tree_pipe, tree_params = FINAL["Decision tree (pruned)"]
tree = tree_pipe.named_steps["model"]
tree_names = readable_names(tree_pipe.named_steps["prep"].get_feature_names_out())
print("Final pruned tree:", tree_params, "| leaves:", tree.get_n_leaves(), "| depth:", tree.get_depth())
fig, ax = plt.subplots(figsize=(min(3 + 1.7 * tree.get_n_leaves(), 20), 1.9 * tree.get_depth() + 2))
anns = plot_tree(tree, feature_names=tree_names, class_names=["LGG", "GBM"], filled=True, rounded=True, impurity=False,
                 proportion=False, precision=1, fontsize=10, ax=ax, label="all")
plotting.recolor_tree(anns, tree, kind="classifier")   # sklearn's default colours are the opposite of the project's
ax.set_title("Final pruned decision tree for the grade")
plotting.save_figure(fig, "clf_decision_tree")
plt.show()
print(export_text(tree, feature_names=tree_names, decimals=1))
results["tree"] = {"params": tree_params, "n_leaves": int(tree.get_n_leaves()), "depth": int(tree.get_depth())}

path = tree_pipe.named_steps["model"].__class__(random_state=config.SEED, criterion=tree_params.get("criterion", "gini")) \
    .cost_complexity_pruning_path(tree_pipe.named_steps["prep"].transform(X), y)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.step(path.ccp_alphas, path.impurities, where="post", color=plotting.INK)
ax.axvline(tree_params.get("ccp_alpha", 0), color=NEUTRAL, ls=":", lw=1.2)
ax.set_xscale("symlog", linthresh=1e-3)
ax.set_xlim(left=0)
ax.set_xlabel("ccp_alpha (symmetric log scale)"); ax.set_ylabel("Total impurity of the leaves")
ax.set_title("Cost-complexity pruning path (dotted: chosen alpha)")
plotting.save_figure(fig, "clf_pruning_path")
plt.show()

## 10. Permutation importance

Drop in ROC-AUC when one predictor is shuffled, on each outer test fold, with the model tuned on the corresponding training part. Mean and SD over the 10 folds. Correlated predictors share importance: shuffling one of them hurts less if another one carries similar information.

In [ ]:
imp_models = ["Logistic regression", "Random forest", "XGBoost", "SVM"]
imp = pd.DataFrame({m: NESTED[m]["importance"].mean() for m in imp_models})
imp_sd = pd.DataFrame({m: NESTED[m]["importance"].std() for m in imp_models})
imp = imp.loc[imp.mean(axis=1).sort_values(ascending=False).index]
display(imp.head(10))
results["permutation_importance_mean"] = imp.to_dict(orient="index")
reporting.save_table_tex(imp.head(10).rename_axis("Predictor").reset_index(), "clf_permutation_importance", float_format="{:.3f}")

top10 = imp.index[:10][::-1]
fig, ax = plt.subplots(figsize=(7.5, 5))
h = 0.2
for i, m in enumerate(imp_models):
    ax.barh(np.arange(len(top10)) + (i - 1.5) * h, imp.loc[top10, m], xerr=imp_sd.loc[top10, m], height=h,
            color=plotting.CATEGORICAL[i + 2], label=m, error_kw=dict(ecolor=NEUTRAL, lw=0.8))
ax.set_yticks(np.arange(len(top10)), top10)
ax.axvline(0, color=NEUTRAL, lw=1)
ax.set_xlabel("Decrease in ROC-AUC when the predictor is shuffled")
ax.set_title("Permutation importance on the outer test folds")
ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
plotting.save_figure(fig, "clf_permutation_importance")
plt.show()

## 11. Summary

Generated from the results above; saved in `results/logs/05_classification.json`.

In [ ]:
rows_ = [
    ("Majority baseline", f"AUC {ci('Majority baseline', 'AUC')}, accuracy {ci('Majority baseline', 'Accuracy')}"),
    ("Highest AUC", f"{best_auc}: AUC {ci(best_auc, 'AUC')}, balanced accuracy {ci(best_auc, 'Balanced accuracy')}, "
                    f"sensitivity {ci(best_auc, 'Sensitivity')}, specificity {ci(best_auc, 'Specificity')}"),
    ("Models tied with it (overlapping AUC intervals)", ", ".join(ties_auc) if ties_auc else "none"),
    ("Paired AUC differences vs the best", "; ".join(f"{r['Model']}: {r.iloc[1]:+.3f} [{r['CI low']:+.3f}, {r['CI high']:+.3f}]"
                                                    for _, r in paired.iterrows() if r["Model"] != best_auc)),
    ("Logistic regression (final)", f"{lr_params}; {results['logistic']['n_zero']} coefficients set to 0; strongest: "
                                    + ", ".join(f"{i} {v:+.2f}" for i, v in lr_coef.iloc[:3, 0].items())),
    ("Pruned tree (final)", f"{tree_params}; depth {tree.get_depth()}, {tree.get_n_leaves()} leaves"),
    ("Most important predictors (permutation)", ", ".join(f"{i} ({v:.3f})" for i, v in imp.mean(axis=1).head(3).items())),
    ("Extreme probabilities (< 0.02 or > 0.98)", f"Naive Bayes {extreme:.2f}, logistic regression {extreme_lr:.2f}"),
]
final_summary = pd.DataFrame(rows_, columns=["Topic", "Finding"])
with pd.option_context("display.max_colwidth", None):
    display(final_summary)
results["findings"] = dict(rows_)
reporting.save_table_tex(final_summary, "clf_summary", column_format="p{4.5cm}p{10.5cm}")
for name, r in NESTED.items():
    results.setdefault("folds", {})[name] = r["folds"].to_dict(orient="records")
path = reporting.save_json(results, "05_classification")
print("Saved", path.relative_to(config.ROOT))
print("Figures:", sorted(p.name for p in config.FIGURES.glob("clf_*.png")))